# Leakage-aware Smart Waste-Bin Training Pipeline

This notebook is a **reproducible replacement workflow**, not a claim that new scores have already been obtained. The uploaded project ZIP did not include `smart_city_sensor_data.csv`. Put the original CSV beside this notebook before running it. If the raw file cannot be recovered, do not report new model metrics as reproduced.

Key safeguards:
- chronological split before fitting preprocessing/calibration;
- trailing (past-only) smoothing rather than centred smoothing;
- no interpolation that uses future readings;
- training labels are generated only within the training segment;
- test labels are generated separately within the test segment; boundary/incomplete cycles are discarded;
- simulated weight is excluded from the primary model;
- test scores are computed only from models fitted on training data.


In [ ]:
from pathlib import Path
import json, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

DATA_PATH = Path("smart_city_sensor_data.csv")
OUT = Path("artifacts_revised")
OUT.mkdir(exist_ok=True)
FULL_LEVEL = 80.0          # explicit operational choice: 80% of calibrated capacity
TRAIN_FRACTION = 0.80
DROP_THRESHOLD_RAW = 20.0  # collection event heuristic on raw sensor scale
MIN_HOURS = 1
PURGE_HOURS = 24           # conservative buffer; increase if cycle durations require it
BIN_CAPACITY_KG = 200.0    # assumption, not measured by the supplied dataset


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} is missing. Recover the original CSV and place it beside this notebook. "
        "The ZIP supplied for revision contains derived SQLite data but not the raw CSV."
    )

raw = pd.read_csv(DATA_PATH)
print("Raw columns:", list(raw.columns))
print("Rows:", len(raw))


## 1. Load and verify the source

The original notebook assumes 12 columns in a fixed order. Confirm the actual CSV header against the dataset documentation before applying the mapping below. Do not silently rename columns if the source schema differs.

In [ ]:
EXPECTED = ['City','Sensor_ID','Lat','Lon','Timestamp','Sensor_Type',
            'Street_Type','Nearby','Vehicles','kWh','Fill','Noise']
if list(raw.columns) == EXPECTED:
    data = raw.copy()
else:
    # Prefer explicit source column names when available.
    aliases = {c.lower(): c for c in raw.columns}
    required = ['sensor_id','timestamp','fill']
    if not all(k in aliases for k in required):
        raise ValueError(
            "CSV schema does not match the expected 12-column format and does not expose "
            "Sensor_ID/Timestamp/Fill equivalents. Inspect the source header and map columns explicitly."
        )
    data = raw.rename(columns={
        aliases['sensor_id']: 'Sensor_ID',
        aliases['timestamp']: 'Timestamp',
        aliases['fill']: 'Fill'
    }).copy()

data['Timestamp'] = pd.to_datetime(data['Timestamp'], errors='coerce')
if 'Sensor_Type' in data.columns:
    # Source records may include multiple sensor types; keep waste sensors if that label exists.
    waste_mask = data['Sensor_Type'].astype(str).str.contains('waste', case=False, na=False)
    if waste_mask.any():
        data = data.loc[waste_mask].copy()
data = data[['Sensor_ID','Timestamp','Fill']].dropna(subset=['Sensor_ID','Timestamp','Fill'])
data['Fill'] = pd.to_numeric(data['Fill'], errors='coerce')
data = data.dropna(subset=['Fill'])
data = data[(data['Fill'] >= 0) & (data['Fill'] <= 100)]
data = data.drop_duplicates(['Sensor_ID','Timestamp']).sort_values(['Timestamp','Sensor_ID']).reset_index(drop=True)
print(data.head(), data['Timestamp'].min(), data['Timestamp'].max(), data['Sensor_ID'].nunique())


## 2. Chronological boundary and training-only calibration

Split timestamps before calculating the capacity reference. The 95th percentile of training-period raw fill values is used as a calibration reference only if the sensor is known to have a stable scale. This is a practical proxy, not a measured physical capacity. The 80% collection threshold remains an explicit operating assumption.

In [ ]:
unique_times = np.array(sorted(data['Timestamp'].unique()))
cut_index = int(len(unique_times) * TRAIN_FRACTION)
cut_time = pd.Timestamp(unique_times[cut_index])
train_raw = data[data['Timestamp'] < cut_time].copy()
test_raw = data[data['Timestamp'] >= cut_time].copy()

# Training-only scale calibration; freeze it for both segments.
capacity_reference = float(train_raw['Fill'].quantile(0.95))
if not np.isfinite(capacity_reference) or capacity_reference <= 0:
    raise ValueError("Invalid training-only calibration reference.")
print("Cut time:", cut_time)
print("Training-only raw sensor reference:", round(capacity_reference, 3))
print("Collection threshold in calibrated units:", FULL_LEVEL, "%")


## 3. Causal cycle detection and label creation

A trailing median only uses current and past values. We do not interpolate missing values across time because linear interpolation can use a future endpoint. For each segment independently, a collection event is inferred when the trailing smoothed raw reading drops by more than `DROP_THRESHOLD_RAW`. The first partial cycle in each segment is discarded. Labels are created only from threshold crossings within that same segment, so training labels cannot depend on test observations. This is still heuristic: a real collection log is preferable.

In [ ]:
def make_segment(segment, segment_name):
    pieces = []
    for sid, g in segment.sort_values('Timestamp').groupby('Sensor_ID', sort=False):
        g = g.sort_values('Timestamp').copy()
        # Causal smoothing: no centered window and no future interpolation.
        g['smooth_raw'] = g['Fill'].rolling(window=5, min_periods=1).median()
        g['drop'] = g['smooth_raw'].diff() < -DROP_THRESHOLD_RAW
        g['cycle_local'] = g['drop'].fillna(False).cumsum()
        # Calibrate with training-only reference, frozen for test.
        g['fill_pct'] = (g['Fill'] / capacity_reference * 100).clip(0, 100)
        g['smooth_pct'] = (g['smooth_raw'] / capacity_reference * 100).clip(0, 100)
        # Label each cycle only within this segment.
        g['cycle_start'] = g.groupby('cycle_local')['Timestamp'].transform('min')
        g['hours_since_collection'] = (
            (g['Timestamp'] - g['cycle_start']).dt.total_seconds() / 3600
        )
        g['full_time'] = pd.NaT
        for cyc, cg in g.groupby('cycle_local'):
            hits = cg.loc[cg['smooth_pct'] >= FULL_LEVEL, 'Timestamp']
            if len(hits):
                g.loc[cg.index, 'full_time'] = hits.iloc[0]
        g['hours_to_full'] = (
            (pd.to_datetime(g['full_time']) - g['Timestamp']).dt.total_seconds() / 3600
        )
        # Current/past-only rate; never use future fill values.
        g['avg_fill_rate'] = (
            g['fill_pct'].diff() / g['Timestamp'].diff().dt.total_seconds().div(3600)
        )
        g['hour'] = g['Timestamp'].dt.hour
        g['is_weekend'] = (g['Timestamp'].dt.dayofweek >= 5).astype(int)
        g['segment'] = segment_name
        pieces.append(g)
    out = pd.concat(pieces, ignore_index=True)
    # Exclude first cycle, threshold-reached rows, missing/incomplete labels, near-zero elapsed time.
    out = out[(out['cycle_local'] > 0) &
              (out['hours_to_full'] > 0) &
              (out['hours_since_collection'] >= MIN_HOURS)].copy()
    return out

train = make_segment(train_raw, 'train')
test = make_segment(test_raw, 'test')

# Purge rows near the split to reduce risk from delayed collection detection and cycle ambiguity.
purge_start = cut_time - pd.Timedelta(hours=PURGE_HOURS)
purge_end = cut_time + pd.Timedelta(hours=PURGE_HOURS)
train = train[train['Timestamp'] < purge_start].copy()
test = test[test['Timestamp'] >= purge_end].copy()

FEATURES = ['fill_pct','hours_since_collection','avg_fill_rate','hour','is_weekend']
TARGET = 'hours_to_full'
train = train.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES+[TARGET])
test = test.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES+[TARGET])
print("Train rows:", len(train), "| Test rows:", len(test))
print("Train ends:", train['Timestamp'].max(), "| Test starts:", test['Timestamp'].min())
if len(train) < 100 or len(test) < 30:
    raise ValueError("Too few rows after causal labelling/purge. Inspect data coverage and cycle detection.")


## 4. Fit and compare models

The baseline uses only current fill percentage. The main model uses current/past-only features. Ridge scaling is inside a pipeline so the scaler is fitted only on training data. No weight is included because the source has no measured weight column; simulated weight would be a derivative of fill and could create misleading evidence.

In [ ]:
def evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    pred = np.clip(model.predict(x_test), 0, None)
    return {
        'MAE_h': float(mean_absolute_error(y_test, pred)),
        'RMSE_h': float(mean_squared_error(y_test, pred) ** 0.5),
        'R2': float(r2_score(y_test, pred)),
    }, pred

ytr, yte = train[TARGET], test[TARGET]
baseline_cols = ['fill_pct']
baseline = LinearRegression()
mlr = LinearRegression()
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

baseline_metrics, baseline_pred = evaluate(baseline, train[baseline_cols], ytr, test[baseline_cols], yte)
mlr_metrics, mlr_pred = evaluate(mlr, train[FEATURES], ytr, test[FEATURES], yte)
ridge_metrics, ridge_pred = evaluate(ridge, train[FEATURES], ytr, test[FEATURES], yte)

results = pd.DataFrame({
    'Fill-only baseline': baseline_metrics,
    'Multiple linear regression': mlr_metrics,
    'Ridge regression': ridge_metrics
}).T
results.round(3)


In [ ]:
# Training-only time-series CV; no test observations enter model selection.
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(
    LinearRegression(), train[FEATURES], ytr,
    cv=tscv, scoring='r2'
)
print("Training-only TimeSeriesSplit R²:", np.round(cv_scores, 3))
print("Mean CV R²:", round(float(cv_scores.mean()), 3))

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(yte, mlr_pred, s=8, alpha=.35)
lim = [0, max(float(yte.max()), float(np.max(mlr_pred)))]
ax.plot(lim, lim, 'r--')
ax.set_xlabel('Actual hours until threshold')
ax.set_ylabel('Predicted hours until threshold')
ax.set_title('Chronological test set: actual vs predicted')
fig.tight_layout()
fig.savefig(OUT/'evaluation_revised.png', dpi=150)
plt.show()

metrics = {
    'data_source': 'Verify canonical source URL/version before submission',
    'raw_csv': str(DATA_PATH),
    'rows_train': int(len(train)),
    'rows_test': int(len(test)),
    'cut_time': str(cut_time),
    'purge_hours_each_side': PURGE_HOURS,
    'capacity_reference_raw_train_only': capacity_reference,
    'collection_threshold_pct': FULL_LEVEL,
    'assumed_bin_capacity_kg': BIN_CAPACITY_KG,
    'weight_feature': 'excluded: no measured weight column in source',
    'features': FEATURES,
    'baseline_fill_only': baseline_metrics,
    'mlr': mlr_metrics,
    'ridge': ridge_metrics,
    'training_time_series_cv_r2': [float(x) for x in cv_scores],
    'training_time_series_cv_r2_mean': float(cv_scores.mean()),
    'limitations': [
        'Dataset source/version must be verified and cited directly.',
        'Collection events inferred from fill drops, not confirmed pickup logs.',
        'Threshold is an operational assumption; validate with physical bins.',
        'Results apply to this dataset and split only.'
    ]
}
with open(OUT/'metrics_revised.json','w') as f:
    json.dump(metrics, f, indent=2)
results.to_csv(OUT/'model_comparison_revised.csv')
print(json.dumps(metrics, indent=2))


## 5. Checks before submission

1. Recover the raw CSV and verify its canonical dataset page, author, version, licence, and date accessed.
2. Run every cell from a clean kernel and save the generated metrics and plot.
3. Copy the new MAE, RMSE and R² into the report and app; do not reuse the legacy values unless they are reproduced.
4. Report the exact split timestamp, purge duration, rows used, and number of cycles retained.
5. Replace the individual reflection template with tasks you personally completed and a specific challenge you solved.
6. If the original CSV cannot be recovered, disclose that limitation and do not claim a raw-data rerun.